# Phase 2 - Notebook 01: SLAM Basics Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase2/01_slam_basics.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand Visual Odometry (VO) fundamentals
2. Compare Feature-based vs Direct methods
3. Learn Bundle Adjustment basics
4. Understand Loop Closure concepts
5. Learn about Keyframe selection strategies

**Estimated Time**: 60 minutes

**Prerequisites**: Phase 1 completion, basic linear algebra

---

## 1. What is SLAM?

**SLAM (Simultaneous Localization and Mapping)** solves two interdependent problems:

1. **Localization**: Where am I? (estimate camera/robot pose)
2. **Mapping**: What does the world look like? (build environment representation)

The challenge: You need a map to localize, but you need to know your location to build a map!

### SLAM vs Related Problems

| Problem | Input | Output | Map Known? |
|---------|-------|--------|------------|
| **Localization** | Sensor + Map | Pose | Yes |
| **Mapping** | Sensor + Poses | Map | Poses known |
| **SLAM** | Sensor only | Pose + Map | No |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch
import torch.nn.functional as F
from typing import List, Tuple, Optional, Dict

# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("SLAM Basics Tutorial")
print("=" * 40)

## 2. Visual Odometry (VO)

**Visual Odometry** estimates camera motion from sequential images. It's the "tracking" part of Visual SLAM.

### VO Pipeline

```
Frame t-1    Frame t
   │            │
   ▼            ▼
┌──────┐    ┌──────┐
│ I_t-1│    │  I_t │
└──┬───┘    └──┬───┘
   │           │
   └─────┬─────┘
         │
         ▼
   Feature Match /
   Direct Alignment
         │
         ▼
   Relative Pose
      T_{t-1,t}
         │
         ▼
   T_t = T_{t-1} × T_{t-1,t}
```

In [ ]:
# Simulate a simple visual odometry scenario

def create_camera_trajectory(n_frames: int = 20) -> np.ndarray:
    """
    Create a simulated camera trajectory (circular path).
    
    Returns:
        positions: [n_frames, 3] array of camera positions
    """
    t = np.linspace(0, 2 * np.pi, n_frames)
    radius = 3.0
    
    x = radius * np.cos(t)
    y = radius * np.sin(t)
    z = np.zeros_like(t)  # Keep camera at same height
    
    return np.stack([x, y, z], axis=1)


def compute_relative_pose(
    pos1: np.ndarray, 
    pos2: np.ndarray,
    look_at: np.ndarray = np.array([0, 0, 0])
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute relative translation and rotation between two positions.
    
    Returns:
        translation: [3] relative translation
        rotation_angle: rotation angle in degrees
    """
    translation = pos2 - pos1
    
    # Simple rotation: angle between look directions
    dir1 = look_at - pos1
    dir2 = look_at - pos2
    dir1 = dir1 / np.linalg.norm(dir1)
    dir2 = dir2 / np.linalg.norm(dir2)
    
    cos_angle = np.clip(np.dot(dir1, dir2), -1, 1)
    angle_deg = np.degrees(np.arccos(cos_angle))
    
    return translation, angle_deg


# Create trajectory
trajectory = create_camera_trajectory(20)

# Compute odometry (relative poses)
translations = []
rotations = []

for i in range(1, len(trajectory)):
    trans, rot = compute_relative_pose(trajectory[i-1], trajectory[i])
    translations.append(np.linalg.norm(trans))
    rotations.append(rot)

print(f"Trajectory: {len(trajectory)} frames")
print(f"Mean translation: {np.mean(translations):.3f} m")
print(f"Mean rotation: {np.mean(rotations):.2f} degrees")

In [ ]:
# Visualize the trajectory

fig = plt.figure(figsize=(12, 5))

# 3D view
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2], 'b-', linewidth=2)
ax1.scatter(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2], c=np.arange(len(trajectory)), cmap='viridis', s=50)
ax1.scatter([0], [0], [0], c='red', s=100, marker='*', label='Origin (look-at point)')

# Draw camera orientations
for i in range(0, len(trajectory), 3):
    pos = trajectory[i]
    direction = -pos / np.linalg.norm(pos) * 0.5  # Looking at origin
    ax1.quiver(pos[0], pos[1], pos[2], direction[0], direction[1], direction[2], 
               color='green', arrow_length_ratio=0.3)

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')
ax1.set_title('Camera Trajectory (3D)')
ax1.legend()

# Top-down view
ax2 = fig.add_subplot(122)
ax2.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=2)
scatter = ax2.scatter(trajectory[:, 0], trajectory[:, 1], c=np.arange(len(trajectory)), cmap='viridis', s=50)
ax2.scatter([0], [0], c='red', s=100, marker='*')
plt.colorbar(scatter, ax=ax2, label='Frame Index')

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_title('Camera Trajectory (Top View)')
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Feature-based vs Direct Methods

Two main approaches to Visual Odometry:

### Feature-based Methods

**Examples**: ORB-SLAM, VINS-Mono, OpenVSLAM

1. Detect keypoints (corners, blobs)
2. Compute descriptors
3. Match across frames
4. Estimate pose from correspondences

**Pros**:
- Robust to illumination changes
- Efficient (sparse computation)
- Works well with large motions

**Cons**:
- Fails in textureless regions
- Loses fine detail

### Direct Methods

**Examples**: LSD-SLAM, DSO, SVO

1. Use pixel intensities directly
2. Minimize photometric error
3. No explicit feature detection

**Pros**:
- Works in low-texture regions
- More precise
- Dense reconstruction possible

**Cons**:
- Sensitive to illumination changes
- Requires good initialization
- Limited motion range

In [ ]:
# Demonstration: Feature-based matching

def create_synthetic_image(size: int = 64, n_features: int = 20) -> Tuple[np.ndarray, np.ndarray]:
    """
    Create a synthetic image with feature points.
    
    Returns:
        image: [size, size] grayscale image
        feature_positions: [n_features, 2] feature locations
    """
    image = np.random.rand(size, size) * 0.3  # Background noise
    
    # Random feature positions
    positions = np.random.randint(5, size-5, (n_features, 2))
    
    # Add bright spots at feature locations
    for y, x in positions:
        image[y-2:y+3, x-2:x+3] += 0.5
    
    return np.clip(image, 0, 1), positions


def transform_features(
    features: np.ndarray, 
    translation: np.ndarray, 
    noise_std: float = 1.0
) -> np.ndarray:
    """
    Apply transformation to features with noise.
    """
    transformed = features + translation
    noise = np.random.randn(*features.shape) * noise_std
    return transformed + noise


# Create two "frames"
img1, features1 = create_synthetic_image()
true_translation = np.array([5, 3])  # Known motion
features2 = transform_features(features1, true_translation, noise_std=0.5)

# Estimate translation from correspondences
estimated_translation = np.mean(features2 - features1, axis=0)

print(f"True translation: {true_translation}")
print(f"Estimated translation: {estimated_translation}")
print(f"Error: {np.linalg.norm(estimated_translation - true_translation):.3f} pixels")

In [ ]:
# Visualize feature matching

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Frame 1
axes[0].imshow(img1, cmap='gray')
axes[0].scatter(features1[:, 1], features1[:, 0], c='red', s=50, marker='x')
axes[0].set_title('Frame t-1 (Features in red)')

# Frame 2 (translated)
img2 = np.roll(img1, true_translation[1], axis=1)
img2 = np.roll(img2, true_translation[0], axis=0)
axes[1].imshow(img2, cmap='gray')
axes[1].scatter(features2[:, 1], features2[:, 0], c='blue', s=50, marker='x')
axes[1].set_title('Frame t (Features in blue)')

# Correspondence visualization
axes[2].imshow(np.zeros((64, 64)), cmap='gray')
for i in range(len(features1)):
    axes[2].plot([features1[i, 1], features2[i, 1]], 
                 [features1[i, 0], features2[i, 0]], 
                 'g-', alpha=0.5)
axes[2].scatter(features1[:, 1], features1[:, 0], c='red', s=30, marker='o', label='Frame t-1')
axes[2].scatter(features2[:, 1], features2[:, 0], c='blue', s=30, marker='o', label='Frame t')
axes[2].set_title('Feature Correspondences')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Demonstration: Direct method (photometric alignment)

def photometric_loss(
    img1: torch.Tensor, 
    img2: torch.Tensor, 
    translation: torch.Tensor
) -> torch.Tensor:
    """
    Compute photometric loss between img1 and translated img2.
    
    Args:
        img1: Reference image [H, W]
        img2: Target image [H, W]
        translation: [2] (tx, ty) translation to apply to img2
    
    Returns:
        L1 photometric loss
    """
    H, W = img1.shape
    
    # Create sampling grid
    y, x = torch.meshgrid(
        torch.linspace(-1, 1, H),
        torch.linspace(-1, 1, W),
        indexing='ij'
    )
    grid = torch.stack([x, y], dim=-1)  # [H, W, 2]
    
    # Apply translation (normalized coordinates)
    tx_norm = 2 * translation[0] / W
    ty_norm = 2 * translation[1] / H
    grid_shifted = grid + torch.stack([tx_norm, ty_norm])
    
    # Sample from img2
    img2_sampled = F.grid_sample(
        img2.unsqueeze(0).unsqueeze(0),
        grid_shifted.unsqueeze(0),
        mode='bilinear',
        padding_mode='zeros',
        align_corners=True
    ).squeeze()
    
    # Compute loss (ignoring border regions)
    border = 5
    diff = torch.abs(img1[border:-border, border:-border] - img2_sampled[border:-border, border:-border])
    return diff.mean()


# Create test images
img1_torch = torch.from_numpy(img1).float()
img2_torch = torch.from_numpy(img2).float()

# Optimize translation using gradient descent
estimated_trans = torch.zeros(2, requires_grad=True)
optimizer = torch.optim.Adam([estimated_trans], lr=0.5)

losses = []
for i in range(100):
    optimizer.zero_grad()
    loss = photometric_loss(img1_torch, img2_torch, estimated_trans)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"True translation: {true_translation}")
print(f"Estimated (direct): {estimated_trans.detach().numpy()}")
print(f"Final loss: {losses[-1]:.4f}")

In [ ]:
# Plot convergence
plt.figure(figsize=(10, 4))

plt.subplot(121)
plt.plot(losses)
plt.xlabel('Iteration')
plt.ylabel('Photometric Loss')
plt.title('Direct Method Convergence')
plt.grid(True, alpha=0.3)

plt.subplot(122)
methods = ['Feature-based', 'Direct Method']
errors = [
    np.linalg.norm(estimated_translation - true_translation),
    np.linalg.norm(estimated_trans.detach().numpy() - true_translation)
]
plt.bar(methods, errors, color=['blue', 'orange'])
plt.ylabel('Translation Error (pixels)')
plt.title('Method Comparison')

plt.tight_layout()
plt.show()

## 4. Camera Pose Representation

### SE(3) - Special Euclidean Group

Camera pose is represented as a **rigid body transformation**:

$$T = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix} \in SE(3)$$

Where:
- $R \in SO(3)$ is a 3x3 rotation matrix
- $t \in \mathbb{R}^3$ is translation

### Rotation Representations

| Representation | Parameters | Pros | Cons |
|----------------|------------|------|------|
| Rotation matrix | 9 | Unique, easy composition | Constrained (orthogonal) |
| Quaternion | 4 | Compact, interpolation | Unit norm constraint |
| Axis-angle | 3 | Minimal | Singularities |
| Euler angles | 3 | Intuitive | Gimbal lock |
| 6D continuous | 6 | Good for optimization | Overcomplete |

In [ ]:
# Rotation representations

def rotation_matrix_from_axis_angle(axis: np.ndarray, angle: float) -> np.ndarray:
    """
    Create rotation matrix from axis-angle representation.
    Uses Rodrigues' formula.
    
    Args:
        axis: [3] unit rotation axis
        angle: rotation angle in radians
    
    Returns:
        R: [3, 3] rotation matrix
    """
    axis = axis / np.linalg.norm(axis)
    K = np.array([
        [0, -axis[2], axis[1]],
        [axis[2], 0, -axis[0]],
        [-axis[1], axis[0], 0]
    ])
    R = np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)
    return R


def quaternion_from_rotation_matrix(R: np.ndarray) -> np.ndarray:
    """
    Convert rotation matrix to quaternion [w, x, y, z].
    """
    trace = np.trace(R)
    if trace > 0:
        s = 0.5 / np.sqrt(trace + 1.0)
        w = 0.25 / s
        x = (R[2, 1] - R[1, 2]) * s
        y = (R[0, 2] - R[2, 0]) * s
        z = (R[1, 0] - R[0, 1]) * s
    else:
        if R[0, 0] > R[1, 1] and R[0, 0] > R[2, 2]:
            s = 2.0 * np.sqrt(1.0 + R[0, 0] - R[1, 1] - R[2, 2])
            w = (R[2, 1] - R[1, 2]) / s
            x = 0.25 * s
            y = (R[0, 1] + R[1, 0]) / s
            z = (R[0, 2] + R[2, 0]) / s
        elif R[1, 1] > R[2, 2]:
            s = 2.0 * np.sqrt(1.0 + R[1, 1] - R[0, 0] - R[2, 2])
            w = (R[0, 2] - R[2, 0]) / s
            x = (R[0, 1] + R[1, 0]) / s
            y = 0.25 * s
            z = (R[1, 2] + R[2, 1]) / s
        else:
            s = 2.0 * np.sqrt(1.0 + R[2, 2] - R[0, 0] - R[1, 1])
            w = (R[1, 0] - R[0, 1]) / s
            x = (R[0, 2] + R[2, 0]) / s
            y = (R[1, 2] + R[2, 1]) / s
            z = 0.25 * s
    return np.array([w, x, y, z])


# Example: Create a rotation and show different representations
axis = np.array([0, 0, 1])  # Z-axis
angle = np.radians(45)  # 45 degrees

R = rotation_matrix_from_axis_angle(axis, angle)
q = quaternion_from_rotation_matrix(R)

print("Rotation Matrix:")
print(R)
print(f"\nQuaternion [w, x, y, z]: {q}")
print(f"Axis-angle: axis={axis}, angle={np.degrees(angle):.1f} degrees")
print(f"\nQuaternion norm: {np.linalg.norm(q):.6f} (should be 1.0)")
print(f"det(R) = {np.linalg.det(R):.6f} (should be 1.0)")

In [ ]:
# SE(3) pose composition

def make_se3(R: np.ndarray, t: np.ndarray) -> np.ndarray:
    """Create SE(3) matrix from rotation and translation."""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t
    return T


def decompose_se3(T: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Extract rotation and translation from SE(3) matrix."""
    return T[:3, :3], T[:3, 3]


# Create two poses
R1 = rotation_matrix_from_axis_angle([0, 0, 1], np.radians(30))
t1 = np.array([1, 0, 0])
T1 = make_se3(R1, t1)

R2 = rotation_matrix_from_axis_angle([0, 0, 1], np.radians(20))
t2 = np.array([0.5, 0.5, 0])
T2 = make_se3(R2, t2)

# Compose: T_composed = T1 @ T2 (apply T2 then T1)
T_composed = T1 @ T2

# Relative pose: T_rel = T1^{-1} @ T2
T_rel = np.linalg.inv(T1) @ T2

print("T1 (30° rotation + [1,0,0] translation):")
print(T1)
print("\nT2 (20° rotation + [0.5,0.5,0] translation):")
print(T2)
print("\nT_composed = T1 @ T2:")
print(T_composed)
print("\nT_relative = T1^{-1} @ T2:")
print(T_rel)

## 5. Bundle Adjustment

**Bundle Adjustment (BA)** is the joint optimization of:
- Camera poses
- 3D point positions

To minimize **reprojection error**:

$$\min_{\{T_i\}, \{X_j\}} \sum_{i,j} \rho\left(\|\pi(T_i, X_j) - x_{ij}\|^2\right)$$

Where:
- $T_i$: Camera pose $i$
- $X_j$: 3D point $j$
- $x_{ij}$: Observed 2D point
- $\pi$: Projection function
- $\rho$: Robust loss (e.g., Huber)

### BA Structure

```
             X1    X2    X3    X4    (3D points)
             │     │     │     │
    C1 ──────●─────●─────┼─────┼─────  (Camera 1 sees X1, X2)
    C2 ──────●─────┼─────●─────┼─────  (Camera 2 sees X1, X3)
    C3 ──────┼─────●─────●─────●─────  (Camera 3 sees X2, X3, X4)
```

In [ ]:
# Simple Bundle Adjustment demonstration

def project_point(
    point_3d: torch.Tensor, 
    pose: torch.Tensor,
    K: torch.Tensor
) -> torch.Tensor:
    """
    Project 3D point to 2D using camera pose and intrinsics.
    
    Args:
        point_3d: [3] 3D point in world coordinates
        pose: [4, 4] world-to-camera transform
        K: [3, 3] camera intrinsics
    
    Returns:
        [2] 2D point in pixel coordinates
    """
    # Transform to camera coordinates
    point_homo = torch.cat([point_3d, torch.ones(1)])
    point_cam = pose @ point_homo
    point_cam = point_cam[:3]
    
    # Project
    point_2d_homo = K @ point_cam
    point_2d = point_2d_homo[:2] / point_2d_homo[2]
    
    return point_2d


def reprojection_error(
    points_3d: torch.Tensor,
    poses: List[torch.Tensor],
    K: torch.Tensor,
    observations: List[Tuple[int, int, torch.Tensor]]  # (camera_idx, point_idx, 2d_obs)
) -> torch.Tensor:
    """
    Compute total reprojection error.
    
    Args:
        points_3d: [N, 3] 3D points
        poses: List of [4, 4] camera poses
        K: [3, 3] camera intrinsics
        observations: List of (camera_idx, point_idx, observed_2d_point)
    
    Returns:
        Total squared reprojection error
    """
    total_error = torch.tensor(0.0)
    
    for cam_idx, point_idx, obs_2d in observations:
        projected = project_point(points_3d[point_idx], poses[cam_idx], K)
        error = torch.sum((projected - obs_2d) ** 2)
        total_error = total_error + error
    
    return total_error


# Create synthetic BA problem
n_cameras = 5
n_points = 10

# True 3D points (random in a cube)
true_points = torch.rand(n_points, 3) * 2 - 1  # [-1, 1]^3

# Camera intrinsics
K = torch.tensor([
    [500.0, 0.0, 320.0],
    [0.0, 500.0, 240.0],
    [0.0, 0.0, 1.0]
])

# True camera poses (cameras looking at origin from different angles)
true_poses = []
for i in range(n_cameras):
    angle = 2 * np.pi * i / n_cameras
    R = rotation_matrix_from_axis_angle([0, 1, 0], angle)
    t = np.array([3 * np.sin(angle), 0, 3 * np.cos(angle)])
    T = make_se3(R, t)
    true_poses.append(torch.from_numpy(T).float())

# Generate observations (with noise)
observations = []
noise_std = 1.0  # pixels

for cam_idx in range(n_cameras):
    for point_idx in range(n_points):
        # Check visibility (simplified: always visible)
        true_2d = project_point(true_points[point_idx], true_poses[cam_idx], K)
        noisy_2d = true_2d + torch.randn(2) * noise_std
        observations.append((cam_idx, point_idx, noisy_2d))

print(f"BA Problem: {n_cameras} cameras, {n_points} points, {len(observations)} observations")

In [ ]:
# Optimize points only (fixed cameras)

# Initialize with noisy points
estimated_points = true_points.clone() + torch.randn_like(true_points) * 0.2
estimated_points.requires_grad_(True)

optimizer = torch.optim.Adam([estimated_points], lr=0.01)

losses = []
for i in range(200):
    optimizer.zero_grad()
    loss = reprojection_error(estimated_points, true_poses, K, observations)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Compute final error
point_error = torch.norm(estimated_points - true_points, dim=1).mean().item()

print(f"Initial reprojection error: {losses[0]:.2f}")
print(f"Final reprojection error: {losses[-1]:.2f}")
print(f"Mean 3D point error: {point_error:.4f} units")

In [ ]:
# Visualize BA results

fig = plt.figure(figsize=(14, 5))

# Loss curve
ax1 = fig.add_subplot(131)
ax1.semilogy(losses)
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Reprojection Error (log)')
ax1.set_title('Bundle Adjustment Convergence')
ax1.grid(True, alpha=0.3)

# 3D view of points and cameras
ax2 = fig.add_subplot(132, projection='3d')

# True points
true_pts = true_points.detach().numpy()
ax2.scatter(true_pts[:, 0], true_pts[:, 1], true_pts[:, 2], 
           c='green', s=50, marker='o', label='True points')

# Estimated points
est_pts = estimated_points.detach().numpy()
ax2.scatter(est_pts[:, 0], est_pts[:, 1], est_pts[:, 2], 
           c='red', s=50, marker='x', label='Estimated points')

# Camera positions
for i, pose in enumerate(true_poses):
    R, t = decompose_se3(pose.numpy())
    cam_pos = -R.T @ t
    ax2.scatter(cam_pos[0], cam_pos[1], cam_pos[2], 
               c='blue', s=100, marker='^')

ax2.set_xlabel('X')
ax2.set_ylabel('Y')
ax2.set_zlabel('Z')
ax2.set_title('3D Reconstruction')
ax2.legend()

# Per-point error
ax3 = fig.add_subplot(133)
point_errors = torch.norm(estimated_points - true_points, dim=1).detach().numpy()
ax3.bar(range(n_points), point_errors)
ax3.set_xlabel('Point Index')
ax3.set_ylabel('3D Error')
ax3.set_title('Per-Point Error')
ax3.axhline(y=np.mean(point_errors), color='r', linestyle='--', label=f'Mean: {np.mean(point_errors):.3f}')
ax3.legend()

plt.tight_layout()
plt.show()

## 6. Loop Closure

**Loop Closure** detects when the camera revisits a previously seen location.

### Why It Matters

Visual Odometry accumulates drift over time. Loop closure:
1. **Detects** revisited places
2. **Corrects** accumulated drift
3. **Creates** globally consistent map

### Loop Closure Pipeline

```
Current Frame
     │
     ▼
┌──────────────┐
│ Place        │     ┌──────────────┐
│ Recognition  │ ◄── │ Keyframe DB  │
└──────┬───────┘     └──────────────┘
       │
       ▼ Candidate matches
┌──────────────┐
│ Geometric    │
│ Verification │
└──────┬───────┘
       │
       ▼ Verified loop
┌──────────────┐
│ Pose Graph   │
│ Optimization │
└──────────────┘
```

In [ ]:
# Simulate drift and loop closure

def simulate_odometry_with_drift(
    true_trajectory: np.ndarray,
    drift_rate: float = 0.02
) -> np.ndarray:
    """
    Simulate odometry with accumulating drift.
    
    Args:
        true_trajectory: [N, 3] true positions
        drift_rate: drift per frame as fraction of motion
    
    Returns:
        [N, 3] estimated trajectory with drift
    """
    estimated = np.zeros_like(true_trajectory)
    estimated[0] = true_trajectory[0]
    
    accumulated_drift = np.zeros(3)
    
    for i in range(1, len(true_trajectory)):
        # True relative motion
        true_motion = true_trajectory[i] - true_trajectory[i-1]
        
        # Add drift
        drift = np.random.randn(3) * drift_rate * np.linalg.norm(true_motion)
        accumulated_drift += drift
        
        # Estimated position
        estimated[i] = true_trajectory[i] + accumulated_drift
    
    return estimated


def detect_loop_closure(
    trajectory: np.ndarray,
    min_frame_gap: int = 10,
    distance_threshold: float = 0.5
) -> List[Tuple[int, int]]:
    """
    Simple loop closure detection based on position proximity.
    
    Returns:
        List of (frame_i, frame_j) loop pairs
    """
    loops = []
    n = len(trajectory)
    
    for i in range(n):
        for j in range(i + min_frame_gap, n):
            dist = np.linalg.norm(trajectory[i] - trajectory[j])
            if dist < distance_threshold:
                loops.append((i, j))
    
    return loops


# Create circular trajectory (loop)
n_frames = 50
true_traj = create_camera_trajectory(n_frames)

# Simulate odometry with drift
estimated_traj = simulate_odometry_with_drift(true_traj, drift_rate=0.03)

# Detect loops in true trajectory
true_loops = detect_loop_closure(true_traj, min_frame_gap=n_frames-5, distance_threshold=0.5)

print(f"Trajectory: {n_frames} frames")
print(f"Final drift: {np.linalg.norm(estimated_traj[-1] - true_traj[-1]):.3f} m")
print(f"Loop closures detected: {len(true_loops)}")

In [ ]:
# Visualize drift and loop closure

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before loop closure
ax1 = axes[0]
ax1.plot(true_traj[:, 0], true_traj[:, 1], 'g-', linewidth=2, label='Ground Truth')
ax1.plot(estimated_traj[:, 0], estimated_traj[:, 1], 'r--', linewidth=2, label='Odometry (with drift)')
ax1.scatter(true_traj[0, 0], true_traj[0, 1], c='green', s=100, marker='s', zorder=5)
ax1.scatter(true_traj[-1, 0], true_traj[-1, 1], c='green', s=100, marker='o', zorder=5)
ax1.scatter(estimated_traj[-1, 0], estimated_traj[-1, 1], c='red', s=100, marker='x', zorder=5)

# Show drift
ax1.annotate('', xy=(estimated_traj[-1, 0], estimated_traj[-1, 1]),
            xytext=(true_traj[-1, 0], true_traj[-1, 1]),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax1.text(0, 3.5, f'Accumulated Drift: {np.linalg.norm(estimated_traj[-1] - true_traj[-1]):.2f} m',
        fontsize=12, ha='center')

ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('Before Loop Closure')
ax1.legend()
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# After loop closure (simplified correction)
ax2 = axes[1]

# Simple loop closure correction: distribute error over trajectory
if true_loops:
    loop_start, loop_end = true_loops[0] if true_loops else (0, n_frames-1)
    loop_error = estimated_traj[loop_end] - true_traj[loop_end]
    
    # Linear interpolation of correction
    corrected_traj = estimated_traj.copy()
    for i in range(loop_start, loop_end + 1):
        alpha = (i - loop_start) / (loop_end - loop_start)
        corrected_traj[i] = estimated_traj[i] - alpha * loop_error
else:
    corrected_traj = estimated_traj.copy()

ax2.plot(true_traj[:, 0], true_traj[:, 1], 'g-', linewidth=2, label='Ground Truth')
ax2.plot(corrected_traj[:, 0], corrected_traj[:, 1], 'b-', linewidth=2, label='After Loop Closure')
ax2.scatter(true_traj[0, 0], true_traj[0, 1], c='green', s=100, marker='s', zorder=5)
ax2.scatter(true_traj[-1, 0], true_traj[-1, 1], c='green', s=100, marker='o', zorder=5)

# Show loop closure constraint
if true_loops:
    i, j = true_loops[0]
    ax2.plot([true_traj[i, 0], true_traj[j, 0]], 
             [true_traj[i, 1], true_traj[j, 1]], 
             'purple', linewidth=3, linestyle=':', label='Loop Constraint')

ax2.text(0, 3.5, f'Corrected Drift: {np.linalg.norm(corrected_traj[-1] - true_traj[-1]):.2f} m',
        fontsize=12, ha='center')

ax2.set_xlabel('X (m)')
ax2.set_ylabel('Y (m)')
ax2.set_title('After Loop Closure')
ax2.legend()
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Keyframe Selection

Not all frames need to be processed/stored. **Keyframes** are selected frames that:
- Have sufficient motion from previous keyframe
- Provide good coverage of the scene
- Are used for optimization (BA, loop closure)

### Selection Criteria

| Criterion | Threshold | Purpose |
|-----------|-----------|----------|
| Translation | > 0.05m | Ensure parallax |
| Rotation | > 5° | New viewpoint |
| Feature overlap | 30-70% | Tracking continuity |
| Tracking quality | Low loss | Reliable frame |

In [ ]:
# Keyframe selection demonstration

def select_keyframes(
    poses: List[np.ndarray],
    min_translation: float = 0.1,
    min_rotation_deg: float = 10.0
) -> List[int]:
    """
    Select keyframes based on motion criteria.
    
    Args:
        poses: List of [4, 4] SE(3) poses
        min_translation: Minimum translation (meters)
        min_rotation_deg: Minimum rotation (degrees)
    
    Returns:
        List of keyframe indices
    """
    keyframes = [0]  # First frame is always a keyframe
    last_kf_pose = poses[0]
    
    for i in range(1, len(poses)):
        current_pose = poses[i]
        
        # Compute relative pose
        relative = current_pose @ np.linalg.inv(last_kf_pose)
        
        # Translation
        translation = np.linalg.norm(relative[:3, 3])
        
        # Rotation (from trace)
        R = relative[:3, :3]
        trace = np.trace(R)
        cos_angle = np.clip((trace - 1) / 2, -1, 1)
        rotation_deg = np.degrees(np.arccos(cos_angle))
        
        # Check criteria
        if translation > min_translation or rotation_deg > min_rotation_deg:
            keyframes.append(i)
            last_kf_pose = current_pose
    
    return keyframes


# Create poses from trajectory
poses = []
for i in range(len(true_traj)):
    # Look at origin
    pos = true_traj[i]
    direction = -pos / np.linalg.norm(pos)
    
    # Simple rotation: face the direction of motion
    z_axis = direction
    y_axis = np.array([0, 0, 1])
    x_axis = np.cross(y_axis, z_axis)
    x_axis = x_axis / np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis, x_axis)
    
    R = np.stack([x_axis, y_axis, z_axis], axis=1)
    T = make_se3(R, pos)
    poses.append(T)

# Select keyframes with different thresholds
kf_strict = select_keyframes(poses, min_translation=0.5, min_rotation_deg=15)
kf_loose = select_keyframes(poses, min_translation=0.2, min_rotation_deg=5)

print(f"Total frames: {len(poses)}")
print(f"Keyframes (strict): {len(kf_strict)} ({len(kf_strict)/len(poses)*100:.1f}%)")
print(f"Keyframes (loose): {len(kf_loose)} ({len(kf_loose)/len(poses)*100:.1f}%)")

In [ ]:
# Visualize keyframe selection

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, kf_indices, title in zip(axes, [kf_strict, kf_loose], 
                                  ['Strict Selection (0.5m, 15°)', 'Loose Selection (0.2m, 5°)']):
    # All frames
    ax.plot(true_traj[:, 0], true_traj[:, 1], 'b-', linewidth=1, alpha=0.3, label='All frames')
    ax.scatter(true_traj[:, 0], true_traj[:, 1], c='blue', s=10, alpha=0.3)
    
    # Keyframes
    kf_traj = true_traj[kf_indices]
    ax.plot(kf_traj[:, 0], kf_traj[:, 1], 'r-', linewidth=2, label=f'Keyframes ({len(kf_indices)})')
    ax.scatter(kf_traj[:, 0], kf_traj[:, 1], c='red', s=80, zorder=5)
    
    # Mark first and last
    ax.scatter(true_traj[0, 0], true_traj[0, 1], c='green', s=150, marker='s', zorder=6, label='Start')
    ax.scatter(true_traj[-1, 0], true_traj[-1, 1], c='purple', s=150, marker='o', zorder=6, label='End')
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title(title)
    ax.legend()
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Covisibility Graph

The **Covisibility Graph** connects keyframes that observe common 3D points.

### Structure

- **Nodes**: Keyframes
- **Edges**: Weighted by number of shared observations
- **Usage**: 
  - Local BA: Optimize nearby keyframes
  - Loop closure: Find candidates
  - Map culling: Remove redundant keyframes

In [ ]:
# Build a simple covisibility graph

def build_covisibility_graph(
    keyframe_positions: np.ndarray,
    distance_threshold: float = 1.5
) -> Dict[int, List[int]]:
    """
    Build covisibility graph based on spatial proximity.
    (Simplified: real systems use shared point observations)
    
    Returns:
        Dict mapping keyframe index to list of covisible keyframes
    """
    n = len(keyframe_positions)
    graph = {i: [] for i in range(n)}
    
    for i in range(n):
        for j in range(i + 1, n):
            dist = np.linalg.norm(keyframe_positions[i] - keyframe_positions[j])
            if dist < distance_threshold:
                graph[i].append(j)
                graph[j].append(i)
    
    return graph


# Build graph from keyframes
kf_positions = true_traj[kf_loose]
covis_graph = build_covisibility_graph(kf_positions, distance_threshold=1.5)

# Statistics
n_edges = sum(len(neighbors) for neighbors in covis_graph.values()) // 2
avg_degree = np.mean([len(neighbors) for neighbors in covis_graph.values()])

print(f"Covisibility Graph:")
print(f"  Nodes (keyframes): {len(covis_graph)}")
print(f"  Edges: {n_edges}")
print(f"  Average degree: {avg_degree:.1f}")

# Show sample connections
print(f"\nSample connections:")
for i in range(min(5, len(covis_graph))):
    print(f"  KF {i} -> {covis_graph[i]}")

In [ ]:
# Visualize covisibility graph

plt.figure(figsize=(10, 10))

# Draw edges
for i, neighbors in covis_graph.items():
    for j in neighbors:
        if j > i:  # Draw each edge once
            plt.plot([kf_positions[i, 0], kf_positions[j, 0]],
                    [kf_positions[i, 1], kf_positions[j, 1]],
                    'gray', linewidth=0.5, alpha=0.5)

# Draw trajectory
plt.plot(kf_positions[:, 0], kf_positions[:, 1], 'b-', linewidth=2, alpha=0.7, label='Trajectory')

# Draw nodes (colored by degree)
degrees = [len(covis_graph[i]) for i in range(len(kf_positions))]
scatter = plt.scatter(kf_positions[:, 0], kf_positions[:, 1], 
                     c=degrees, cmap='viridis', s=100, zorder=5)
plt.colorbar(scatter, label='Degree (# connections)')

# Label some nodes
for i in range(0, len(kf_positions), 3):
    plt.annotate(str(i), (kf_positions[i, 0], kf_positions[i, 1]), 
                fontsize=8, ha='center', va='bottom')

plt.xlabel('X (m)')
plt.ylabel('Y (m)')
plt.title('Covisibility Graph')
plt.legend()
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Connection to 3DGS-SLAM

How do these SLAM concepts apply to 3DGS-based systems?

| SLAM Concept | Traditional | 3DGS-SLAM |
|--------------|-------------|------------|
| **Map** | Points/Features | 3D Gaussians |
| **Tracking** | Feature matching | Render & compare |
| **Mapping** | Triangulation | Gaussian optimization |
| **BA** | Reprojection error | Photometric loss |
| **Loop Closure** | BoW matching | Re-rendering similarity |
| **Keyframes** | Same criteria | + rendering quality |

In [ ]:
# Summary comparison

comparison_table = """
┌────────────────────┬───────────────────────┬─────────────────────────┐
│     Component      │   Traditional SLAM    │       3DGS-SLAM         │
├────────────────────┼───────────────────────┼─────────────────────────┤
│ Map Representation │ Sparse points/mesh    │ 3D Gaussians            │
│ Tracking Loss      │ Feature reprojection  │ Photometric (render)    │
│ Mapping Update     │ Triangulation         │ Gradient-based optim    │
│ Densification      │ Manual triangulation  │ Automatic (gradients)   │
│ Visualization      │ Point cloud/mesh      │ Novel view synthesis    │
│ Real-time capable  │ Yes                   │ Yes (with CUDA)         │
│ Dense output       │ Requires post-proc    │ Native                  │
└────────────────────┴───────────────────────┴─────────────────────────┘
"""
print(comparison_table)

## 10. Summary

### Key Concepts Learned

1. **Visual Odometry**: Estimating camera motion from sequential images
2. **Feature vs Direct**: Trade-offs between robustness and precision
3. **Bundle Adjustment**: Joint optimization of poses and structure
4. **Loop Closure**: Detecting revisited places to correct drift
5. **Keyframe Selection**: Choosing representative frames for efficiency
6. **Covisibility**: Connecting frames with shared observations

### Connection to 3DGS

- 3DGS replaces point-based maps with Gaussian representations
- Tracking uses render-and-compare instead of feature matching
- Mapping optimizes Gaussians using photometric loss
- Same keyframe/covisibility concepts apply

---

## What's Next?

**[02_splatam_architecture.ipynb](./02_splatam_architecture.ipynb)** - Deep dive into SplaTAM's tracking and mapping architecture

---

## References

1. Scaramuzza & Fraundorfer, "Visual Odometry: Part I & II", IEEE RAM 2011
2. Mur-Artal et al., "ORB-SLAM2", IEEE T-RO 2017
3. Engel et al., "LSD-SLAM", ECCV 2014
4. Engel et al., "DSO", IEEE T-PAMI 2018
5. Triggs et al., "Bundle Adjustment: A Modern Synthesis", Vision Algorithms 2000